In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%cd ../../..

/Users/dave/Library/CloudStorage/OneDrive-PolitecnicodiMilano/PhD/Repositories/DT-rse


## Test Adaptation Phase

In this notebook, we test the adaptation phase of the OL routine with an offline dataset generated by the estimation routine. The idea is to validate the method and create a new faulty cluster from the outlier set.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns
from pathlib import Path
from scipy import signal
import plotly.graph_objects as go
from IPython.display import display, clear_output
import time

from ernesto.postprocessing.visualization import ernesto_plotter
from ernesto.adaptation.regime_shift.stats import *

In [3]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# 1. Set the basic style
# 'paper' context sets the base font size small (approx 8-9pt). 
# Scaling it by 1.3-1.5 usually matches standard 11pt/12pt thesis text nicely.
sns.set_context("paper", font_scale=1.4) 

# 2. Define the visual style
sns.set_style('whitegrid', {
    'grid.linestyle': '--', 
    'grid.alpha': 0.5,          # Lower alpha for less intrusive grid
    'grid.color': '.8',
    'axes.edgecolor': '0.15',
    'font.family': 'serif',     # Matches LaTeX default
    'font.serif': ['Times New Roman', 'Computer Modern Roman', 'DejaVu Serif'],
})

# 3. Fine-tune Matplotlib params for Publication Quality
plt.rcParams.update({
    # Figure Size: 6x3.7 is close to the Golden Ratio for A4/Letter width
    'figure.figsize': (6.0, 3.75), 
    
    # Line width: Thicker for the main mean line
    'lines.linewidth': 2.0,     
    
    # Fonts: Ensure math looks like LaTeX
    'mathtext.fontset': 'cm',   # 'cm' = Computer Modern (LaTeX standard)
    'mathtext.rm': 'serif',
    
    # Axes and Ticks
    'axes.linewidth': 1.2,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 4,
    'ytick.major.size': 4,
    'xtick.bottom': True,       # Ensure ticks are visible
    'ytick.left': True,
    
    # Legend: Clean and readable
    'legend.frameon': True,
    'legend.framealpha': 0.9,
    'legend.fancybox': False,   # Square corners match academic style better
    'legend.edgecolor': '0.8',
})

# 4. Color Palette: High contrast, colorblind safe
# Using 'bright' or 'colorblind' is best for distinguishing RL agents
plot_colors = sns.color_palette("colorblind")

In [4]:
folder =folder = "data/output/adaptive/"
folder = folder + 'region_shift_experiment/test_optimization/1_5M_300estimates_new/'
path = Path(folder)

In [5]:
samples_file = 'parameter_evolution.csv'
# ground_file = 'ground_0.csv'
# sim_file = 'dataset_0.csv'

df_samples = pd.read_csv(path / samples_file)
# df_ground = pd.read_csv(path / ground_file)
# df_sim = pd.read_csv(path / sim_file)

In [6]:
df_samples.tail(20)

,soc,temperature,r0,r1,c1,cluster,is_outlier,time_step
294,0.611111,283.173116,0.004180,0.004324,13699.775399,NaN,True,281
295,0.600708,283.173396,0.004219,0.004287,13688.267400,NaN,True,282
296,0.600681,283.173113,0.003794,0.004639,12143.046957,NaN,True,283
297,0.611111,283.173116,0.004180,0.004323,13699.936964,NaN,True,284
298,0.600708,283.173396,0.004216,0.004288,13668.796165,NaN,True,285
299,0.600681,283.173113,0.003794,0.004639,12141.404026,NaN,True,286
300,0.611111,283.173116,0.004180,0.004324,13697.395187,NaN,True,287
301,0.600708,283.173396,0.004218,0.004287,13682.552576,NaN,True,288
302,0.600681,283.173113,0.003793,0.004640,12140.566250,NaN,True,289
303,0.611111,283.173116,0.004180,0.004324,13699.067713,NaN,True,290


## Complete Iterative Routine: affinity + adaptation

In [7]:
params = ['r0', 'r1', 'c1']
options = {
    'min_number_outliers': 20,
    'alpha_ks': 0.01,
    'grid_size': 30,
    'radius': 0.1,
    'max_potential': 1e-3,
    'lambda': 0.8,
    'beta': 1.5,
    'eta': 1e-6,
    'bandwidth': 0.2,
    'n_candidates': 200,
    'k-neighbours': 50,
    'distance': 'maha',
    'threshold_for_maha': 'fisher',
    'alpha_mcd': 0.01,
    'support_fraction': 1.,
    'max_refine_iters': 0
}

In [8]:
def affinity_test(clusters: dict, point: np.ndarray, time: float, alpha: float = 0.05):
    """
    Affinity procedure
    """
    res = {}
    outlier_check = True
    
    for key, clu in clusters.items():
        centroid = np.mean(clu[0], axis=0)
        cov = np.cov(clu[0], rowvar=False)
        
        d, threshold, p, is_outlier = maha2_fisher_threshold(X=point, mean=centroid, cov=cov, n=len(clu[0]), alpha=alpha)
        res[key] = [d[0], p[0], is_outlier[0]]
        
        if not is_outlier[0]:
            clusters[key] = [np.vstack((clusters[key][0], point)), np.append(clusters[key][1], time)]
            outlier_check = False
            return clusters, key
    return {}, None

In [9]:
def adapation_routine(clusters: dict, outliers: np.ndarray, options: dict, times: np.ndarray):
    """
    Adaptation routine
    """
    mu = None
    sigma = None
    support = []
    indices = []
    idx_list = []
    
    if len(outliers) > options['min_number_outliers']:
        # Perform KS tests between each cluster and the outliers    
        ks_result = True
        for idx, cluster in enumerate(clusters.values()):
            _, p = ks_test(cluster[0], outliers)
            
            # If p-value is greater than alpha_ks, we consider the distributions similar (H0 not rejected)
            if p > options['alpha_ks']:  # not significantly different
                ks_result = False
              
        # If all KS tests passed, look for a new cluster
        if ks_result:
            # Use mountain method to find the new cluster centroid and MinCovDet to estimate its covariance
            center, indices, _, _ = trovo_mountain_method(points=outliers, 
                                                          times=times,
                                                          curr_time=times[-1],
                                                          lambda_=options['lambda'],
                                                          radius=options['radius'],
                                                          eta=options['eta'],
                                                          max_iter=1000)
            support = outliers[indices]
            
            if len(support) > 3:
                mu, sigma, idx_list = mcd_custom(points=support, max_iter_factor=10)                                                      
                
                # If we have enough outliers, create a new cluster
                if len(idx_list) > options['min_number_outliers']:
                    clusters['faulty'] = [support[idx_list], times[np.array(indices)[idx_list]]]
                    outliers = np.delete(outliers, np.array(indices)[idx_list], axis=0)
                    times = np.delete(times, np.array(indices)[idx_list], axis=0)
                    return clusters, outliers, times
                    
                else:
                    pass #print(f"ERROR: Not enough points to be included within the faulty cluster with center {center}!")
            else:
                pass #print(f'ERROR: Not enough points in the support of the center {center}')
        else:
            pass #print("ERROR: KS test not passed!")
    else:
        pass #print("ERROR: Not enough outliers samples!")

    return None, None, None

### Iterative 

In [10]:
df_inliers = df_samples[df_samples['is_outlier'] == False]
df_outliers = df_samples[df_samples['is_outlier'] == True]

In [11]:
X_est = df_samples[df_samples['time_step'] > 0]
nom_cluster = [df_samples[df_samples['time_step'] == 0][params].to_numpy(), 
               df_samples[df_samples['time_step'] == 0]['time_step'].to_numpy(),]

In [12]:
clusters = {'nominal': nom_cluster} 
outlier_set = [] 
times_set = [] 

for i in range(len(X_est)): 
    # AFFINITY CHECK 
    x = X_est[params].to_numpy()[i] 
    new_clusters, _ = affinity_test(clusters, point=x, time=X_est['time_step'].to_numpy()[i]) 
    
    if new_clusters: 
        clusters = new_clusters 
    else:
        outlier_set.append(x) 
        times_set.append(X_est['time_step'].to_numpy()[i]) 
        
    # ADAPTATION PROCEDURE 
    new_clusters, new_outlier_set, new_times_set = adapation_routine(clusters=clusters, 
                                                                     outliers=np.array(outlier_set), 
                                                                     options=options, 
                                                                     times=np.array(times_set)) 
    
    if new_clusters is not None: 
        clusters = new_clusters 
    if new_outlier_set is not None: 
        outlier_set = new_outlier_set.tolist() 
    if new_times_set is not None: 
        times_set = new_times_set.tolist()

ValueError: Input vector should be 1-D.

In [13]:
def interactive_plot3d(X_est, 
                       params, 
                       nom_cluster, 
                       options):
    # --- Setup data containers ---
    clusters = {'nominal': nom_cluster}
    outlier_set = []
    times_set = []
    
    # Create initial 3D figure
    fig = go.Figure()
    
    fig.update_layout(
        title="Adaptive Clustering Evolution",
        scene=dict(
            xaxis_title="R0",
            yaxis_title="R1",
            zaxis_title="C1",
        ),
        showlegend=True,
    )
    
    # --- Main adaptive loop ---
    for i in range(len(X_est)):
        # AFFINITY CHECK
        x = X_est[params].to_numpy()[i]
        new_clusters, inlier_key = affinity_test(clusters, point=x, time=X_est['time_step'].to_numpy()[i])
    
        if new_clusters:
            clusters = new_clusters
        else:
            outlier_set.append(x)
            times_set.append(X_est['time_step'].to_numpy()[i])
    
        # ADAPTATION PROCEDURE
        new_clusters, new_outlier_set, new_times_set = adapation_routine(
            clusters=clusters,
            outliers=np.array(outlier_set),
            options=options,
            times=np.array(times_set),
        )
    
        if new_clusters is not None:
            clusters = new_clusters
        if new_outlier_set is not None:
            outlier_set = new_outlier_set.tolist()
        if new_times_set is not None:
            times_set = new_times_set.tolist()
    
        # --- Update visualization ---
        clear_output(wait=True)  # refresh for live effect
        fig.data = []  # clear old traces
    
        # Plot nominal cluster (blue)
        fig.add_trace(go.Scatter3d(
            x=clusters['nominal'][0][:, 0],
            y=clusters['nominal'][0][:, 1],
            z=clusters['nominal'][0][:, 2],
            mode='markers',
            marker=dict(size=4, color='blue', opacity=0.6),
            name='Nominal Cluster'
        ))
    
        # Plot outliers (red)
        if len(outlier_set) > 0:
            outliers_arr = np.array(outlier_set)
            fig.add_trace(go.Scatter3d(
                x=outliers_arr[:, 0],
                y=outliers_arr[:, 1],
                z=outliers_arr[:, 2],
                mode='markers',
                marker=dict(size=5, color='red', symbol='circle'),
                name='Outliers'
            ))
    
        # Plot any new clusters (green, orange, purple…)
        for name, cluster_data in clusters.items():
            if name != 'nominal':
                fig.add_trace(go.Scatter3d(
                    x=cluster_data[0][:, 0],
                    y=cluster_data[0][:, 1],
                    z=cluster_data[0][:, 2],
                    mode='markers',
                    marker=dict(size=4, opacity=0.7),
                    name=f"{name}",
                ))
    
        display(fig)
        time.sleep(0.2)  # small pause for visualization update

In [14]:
interactive_plot3d(
    X_est=X_est,
    params=["r0", "r1", "c1"],
    nom_cluster=nom_cluster,
    options=options,
)

ValueError: Input vector should be 1-D.

In [15]:
import numpy as np
import time
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, clear_output

def interactive_temporal_plot(X_est, params, nom_cluster, options):
    """
    Interactive Plotly visualization with 3 time-dependent subplots:
    one for each parameter, showing nominal cluster, inliers, outliers,
    and new faulty clusters dynamically.
    """
    clusters = {'nominal': nom_cluster}
    outlier_set = []
    times_set = []

    # --- Create subplots (one per dimension) ---
    n_params = len(params)
    fig = go.FigureWidget(make_subplots(
        rows=n_params, 
        cols=1,
        shared_xaxes=True,
        subplot_titles=params,
        vertical_spacing=0.05
    ))

    colors = dict(nominal="blue", inlier="green", outlier="red", faulty="orange")

    # Initialize traces
    for i, param in enumerate(params):
        # Nominal cluster
        fig.add_trace(go.Scatter(
            x=[], y=[], mode='markers', name="Nominal",
            marker=dict(color=colors["nominal"], opacity=0.7)
        ), row=i+1, col=1)

        # Inliers (good points detected online)
        fig.add_trace(go.Scatter(
            x=[], y=[], mode='markers', name="Inliers",
            marker=dict(color=colors["inlier"], opacity=0.7)
        ), row=i+1, col=1)

        # Outliers
        fig.add_trace(go.Scatter(
            x=[], y=[], mode='markers', name="Outliers",
            marker=dict(color=colors["outlier"], opacity=0.8)
        ), row=i+1, col=1)

        # Faulty clusters (detected among outliers)
        fig.add_trace(go.Scatter(
            x=[], y=[], mode='markers', name="Faulty Cluster",
            marker=dict(color=colors["faulty"], opacity=0.8)
        ), row=i+1, col=1)

        fig.update_yaxes(title_text=param, row=i+1, col=1)

    fig.update_xaxes(title_text="Time Step", row=n_params, col=1)
    fig.update_layout(
        title="Adaptive Clustering Temporal Evolution (3 Parameters)",
        showlegend=True,
        height=300 * n_params,
        template="plotly_white"
    )

    # Display once (persistent)
    display(fig)

    # --- Initialize nominal cluster (t=0) ---
    nominal_data = X_est[X_est['time_step'] == 0][params].to_numpy()
    nominal_times = [0] * len(nominal_data)

    for p_idx, param in enumerate(params):
        fig.data[4*p_idx + 0].x = nominal_times
        fig.data[4*p_idx + 0].y = nominal_data[:, p_idx]

    # --- Main adaptive loop ---
    for i in range(len(X_est)):
        t = X_est['time_step'].to_numpy()[i]
        x = X_est[params].to_numpy()[i]

        # 1️⃣ Affinity test: determine if new or known cluster
        new_clusters, inlier_key = affinity_test(clusters, point=x, time=t)
        if new_clusters:
            clusters = new_clusters
        else:
            outlier_set.append(x)
            times_set.append(t)

        # 2️⃣ Adaptation routine (try to build new clusters from outliers)
        new_clusters, new_outlier_set, new_times_set = adapation_routine(
            clusters=clusters,
            outliers=np.array(outlier_set),
            options=options,
            times=np.array(times_set)
        )

        if new_clusters is not None:
            clusters = new_clusters
        if new_outlier_set is not None:
            outlier_set = new_outlier_set.tolist()
        if new_times_set is not None:
            times_set = new_times_set.tolist()

        # 3️⃣ Compute inliers as points that are not outliers
        if inlier_key is not None:
            inlier_data = clusters[inlier_key][0]
            inlier_times = clusters[inlier_key][1]

        # 4️⃣ Update figure traces
        for p_idx, param in enumerate(params):

            if inlier_key is not None:
                if inlier_key == 'nominal':
                    # Inliers (everything seen so far)
                    fig.data[4*p_idx + 1].x = inlier_times
                    fig.data[4*p_idx + 1].y = inlier_data[:, p_idx]
                else:
                    # Faulty clusters (any cluster beyond nominal)
                    for k, clu in clusters.items():
                        if k != 'nominal':
                            fig.data[4*p_idx + 3].x = inlier_times
                            fig.data[4*p_idx + 3].y = inlier_data[:, p_idx]

            # Outliers
            if len(outlier_set) > 0:
                out_arr = np.array(outlier_set)
                fig.data[4*p_idx + 2].x = times_set
                fig.data[4*p_idx + 2].y = out_arr[:, p_idx]

            
        # Update title live
        fig.update_layout(
            title_text=f"Adaptive Clustering — Step {i+1}/{len(X_est)} (t={t})"
        )

        time.sleep(0.2)

In [16]:
interactive_temporal_plot(
    X_est=X_est,
    params=["r0", "r1", "c1"],
    nom_cluster=nom_cluster,
    options=options,
)

FigureWidget({
    'data': [{'marker': {'color': 'blue', 'opacity': 0.7},
              'mode': 'markers',
              'name': 'Nominal',
              'type': 'scatter',
              'uid': '4e94f661-2d29-4994-aa35-ccef81ffba65',
              'x': [],
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'marker': {'color': 'green', 'opacity': 0.7},
              'mode': 'markers',
              'name': 'Inliers',
              'type': 'scatter',
              'uid': '71acd992-8e78-4436-a178-2220ca5feeaf',
              'x': [],
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'marker': {'color': 'red', 'opacity': 0.8},
              'mode': 'markers',
              'name': 'Outliers',
              'type': 'scatter',
              'uid': '9fcd47c4-60a3-4b54-816f-4ccf9813d1d9',
              'x': [],
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'mar

ValueError: Input vector should be 1-D.